# System E: GPT-2 small, poetry-pretrained (Phase 3)

Fully fine-tunes a GPT-2 small (124M) checkpoint in two stages: Stage 1 on the 18,850 training quatrains, then Stage 2
on the 3,382 training disses, starting from Stage 1's best epoch. The prompt carries the same information as Systems A
and D (traits, intensity, the four planned end words), and each line is tagged with its end word first, as in
System D:

```
Traits: lazy, funny
Intensity: harsh
Rhymes: late wait routine scene
Poem:
[late] you live by your clock, not a minute too late,
```

The tag is stripped on decoding (see `generation/gpt2_lm.py`). Nothing forces the end word, so the report measures how
often it lands. The last section compares E with Systems A and D on the same prompts, from D's saved results. The plan
is in `pipeline.md` (Phase 3, System E).

**Which checkpoint:** `MODEL_KEY` is one of `gpt2.models` in `config.yaml`: `poems` (default), `crpo` or `base`. The
checkpoint names follow it (`gen_E_poems_stage1_best/`, `gen_E_poems_stage2_best/`). `STAGE1 = False` skips Stage 1,
for the ablation (`gen_E_poems_nos1_...`). Both can also be set with the `GPT2_MODEL` and `GPT2_STAGE1` environment
variables.

**On Colab:** Runtime → Change runtime type → **T4 GPU**, then Run all. The repo is cloned from GitHub, so push your
changes first. Checkpoints are saved to Google Drive (`MyDrive/CS425/checkpoints/`). A finished stage is not trained
again: if the runtime disconnects, run all again and it continues from the last finished stage.

**Locally:** pick the project's `.venv` as the kernel. `report()` comes from `qwen_lora.py`, which needs `peft`
(`pip install peft`).

**Smoke test:** set `SMOKE_TEST=1` in the environment (or `SMOKE_TEST = True` below): 100 training and 20 validation
examples per stage, 1 epoch each, 2 prompts per sample set; checkpoints go to a `smoke_test` subfolder.

**Expected time:** not measured on a T4 yet. Stage 1 is about 1.5M tokens per epoch and Stage 2 about 0.3M; the
first full run should record the times here.

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/SchoolWork-awork2103/CS425-Project.git"
BRANCH = "feat/pretrained_poem_models"
DRIVE_CKPT_DIR = "/content/drive/MyDrive/CS425/checkpoints"
SMOKE_TEST = os.environ.get("SMOKE_TEST") == "1"     # a few examples only, to check the notebook runs end to end

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    root = "/content/CS425-Project"
    if os.path.exists(root):
        subprocess.run(["git", "-C", root, "pull", "-q"], check=True)
    else:
        subprocess.run(["git", "clone", "-q", "-b", BRANCH, REPO_URL, root], check=True)
    # Colab ships an old torchao that recent peft refuses to load alongside; nothing here uses it
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "accelerate", "transformers>=4.56",
                    "sentencepiece", "pyyaml", "pronouncing", "wordfreq"], check=True)
    os.chdir(root)
    CKPT_DIR = DRIVE_CKPT_DIR
    RUN_DIR = "/content/runs"        # the Trainer's epoch checkpoints stay off Drive; only <name>_best/ is saved there
else:
    if os.path.basename(os.getcwd()) == "generation":
        os.chdir("..")
    CKPT_DIR = "checkpoints"
    RUN_DIR = None
if SMOKE_TEST:
    CKPT_DIR = os.path.join(CKPT_DIR, "smoke_test")
sys.path[:0] = [os.getcwd(), os.path.join(os.getcwd(), "generation")]
print("project:", os.getcwd(), "| checkpoints:", CKPT_DIR, "| smoke test:", SMOKE_TEST)

In [ ]:
import json, math, random, time
import torch
from utils import load_config, project_path, set_seed
from gpt2_lm import encode_all, generate, load_base, load_split, make_collate, read_jsonl, train
from qwen_lora import report          # the same checks as System D

cfg = load_config()
g = cfg["gpt2"]
SEED = set_seed(cfg["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_KEY = os.environ.get("GPT2_MODEL", "poems")       # poems, crpo or base (gpt2.models in config.yaml)
STAGE1 = os.environ.get("GPT2_STAGE1", "1") == "1"      # False: Stage 2 only, for the ablation
MODEL_NAME = g["models"][MODEL_KEY]
NAME = f"gen_E_{MODEL_KEY}" + ("" if STAGE1 else "_nos1")

diss_dir = project_path(cfg["paths"]["diss_dir"])
rows = {s: read_jsonl(diss_dir / f"{s}.jsonl") for s in ("train", "val", "test")}
trait_info = json.load(open(project_path(cfg["paths"]["traits"])))
pairs = {stage: {s: load_split(stage, s, cfg) for s in ("train", "val")} for stage in (1, 2)}
print("device:", device, torch.cuda.get_device_name() if device == "cuda" else "", "| model:", MODEL_NAME,
      "| checkpoints:", NAME)
print({f"stage {st} {s}": len(v) for st, d in pairs.items() for s, v in d.items()})
print("".join(pairs[2]["train"][0]))

In [ ]:
# Hyperparameters (config.yaml, gpt2; pipeline.md, Phase 3, System E)
LR1, LR2 = g["lr"]                   # Stage 1 (quatrains), Stage 2 (disses)
EPOCHS, PATIENCE = g["epochs"], g["patience"]
BATCH_SIZE, ACCUM = g["batch_size"], g["accum"]    # 16 examples per optimizer step
MAX_LEN = g["max_len"]
SAMPLING = dict(temperature=0.8, top_p=0.9, rep_penalty=1.2)    # same as Systems A and D
N_SAMPLES, K_PROMPTS = 4, 40                 # samples per prompt, prompts per set (as in train_qwen.ipynb)

if SMOKE_TEST:
    EPOCHS, N_SAMPLES, K_PROMPTS = 1, 2, 2
    pairs = {stage: {"train": d["train"][:100], "val": d["val"][:20]} for stage, d in pairs.items()}

## Training
Each stage keeps its best epoch by validation loss and stops after `PATIENCE` epochs without improvement. Stage 2
starts from Stage 1's best checkpoint (or from the original checkpoint when `STAGE1` is off).

In [ ]:
def run_stage(stage, init, lr):
    """Trains one stage from init (a Hugging Face name or a checkpoint folder), or loads it if it already finished."""
    name = f"{NAME}_stage{stage}"
    best, hist_path = os.path.join(CKPT_DIR, f"{name}_best"), os.path.join(CKPT_DIR, f"{name}_history.json")
    if os.path.exists(best) and os.path.exists(hist_path):
        print(f"stage {stage}: found {best}, not training it again")
        return json.load(open(hist_path))
    model, tok = load_base(init, device)
    ex = {}
    for s in ("train", "val"):
        ex[s], skipped = encode_all(tok, pairs[stage][s], MAX_LEN)
        print(f"stage {stage} {s}: {len(ex[s])} examples, {skipped} skipped for length")
    if device == "cuda":
        torch.cuda.reset_peak_memory_stats()
    t = time.time()
    hist = train(model, tok, ex["train"], ex["val"], CKPT_DIR, name, lr=lr, epochs=EPOCHS, patience=PATIENCE,
                 batch_size=BATCH_SIZE, accum=ACCUM, seed=SEED, run_root=RUN_DIR)
    print(f"stage {stage}: {len(hist)} epochs in {(time.time() - t) / 60:.1f} min" +
          (f", peak GPU memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GB" if device == "cuda" else ""))
    del model
    torch.cuda.empty_cache()
    return hist

hist = {}
if STAGE1:
    hist["stage1"] = run_stage(1, MODEL_NAME, LR1)
hist["stage2"] = run_stage(2, os.path.join(CKPT_DIR, f"{NAME}_stage1_best") if STAGE1 else MODEL_NAME, LR2)
model, tok = load_base(os.path.join(CKPT_DIR, f"{NAME}_stage2_best"), device)

## Samples
Validation disses (training traits) and held-out traits from the test set: the same prompts and sampling settings
as `train_gru.ipynb` and `train_qwen.ipynb`.

In [ ]:
val_rows = list(rows["val"])
random.Random(SEED).shuffle(val_rows)
heldout_rows = [r for r in rows["test"] if r["kind"] == "heldout"]
random.Random(SEED).shuffle(heldout_rows)
to_prompt = lambda r: {"traits": r["traits"], "intensity": r["intensity"], "end_words": r["end_words"]}
prompt_sets = {"seen": [to_prompt(r) for r in val_rows[:K_PROMPTS]],
               "heldout": [to_prompt(r) for r in heldout_rows[:K_PROMPTS]]}

def sample_e(prompts):
    torch.manual_seed(SEED)
    return [generate(model, tok, p["traits"], p["intensity"], p["end_words"], n=N_SAMPLES, device=device,
                     **SAMPLING) for p in prompts]

t = time.time()
samples_e = {k: sample_e(v) for k, v in prompt_sets.items()}
poems_e = {k: [[s["lines"] for s in ss] for ss in v] for k, v in samples_e.items()}
print(f"sampled {sum(len(v) for v in prompt_sets.values()) * N_SAMPLES} poems in {(time.time() - t) / 60:.1f} min")
reports_e = {}
for k, title in (("seen", "training traits (validation disses)"), ("heldout", "held-out traits (never seen in training)")):
    print(f"\n=== {title}")
    reports_e[k] = report(prompt_sets[k], poems_e[k], trait_info, show=6)

# Habits the pretrained checkpoints may keep after fine-tuning (pipeline.md, System E): E-crpo's <start> line tags, and
# E-poems' literal "\n" line breaks and capital letters. The training poems have none of these.
texts = [s["text"] for v in samples_e.values() for ss in v for s in ss]
for label, has in (("<start> tag", lambda t: "<start>" in t), ('literal "\\n"', lambda t: "\\n" in t),
                   ("capital letters", lambda t: any(c.isupper() for c in t))):
    print(f"{label}: {sum(map(has, texts))} of {len(texts)} samples")

In [ ]:
# Test loss on the diss test set, separately for training traits and held-out traits. Perplexity is per GPT-2 token:
# comparable between the E checkpoints (same tokenizer), not with System A or D.
collate = make_collate(tok.pad_token_id)

@torch.no_grad()
def test_loss(ps):
    ex, _ = encode_all(tok, ps, MAX_LEN)
    model.eval()
    total = count = 0
    for b in torch.utils.data.DataLoader(ex, BATCH_SIZE * 2, collate_fn=collate):
        b = {k: v.to(device) for k, v in b.items()}
        k = (b["labels"][:, 1:] != -100).sum().item()
        total, count = total + model(**b).loss.item() * k, count + k
    return total / count

test_pairs = list(zip(load_split(2, "test", cfg), rows["test"]))
if SMOKE_TEST:
    test_pairs = test_pairs[:20] + [x for x in test_pairs if x[1]["kind"] == "heldout"][:20]
test = {"seen_traits": test_loss([p for p, r in test_pairs if r["kind"] != "heldout"]),
        "heldout_traits": test_loss([p for p, r in test_pairs if r["kind"] == "heldout"])}
for name, loss in test.items():
    print(f"test {name:15s} loss {loss:.4f}  perplexity {math.exp(loss):6.2f}")

## Comparison with Systems A and D
A and D come from `eval/results/gen_D/gen_D_1.5b_summary.json`: the same 40 + 40 prompts, 4 samples each, with the
same sampling settings and checks, sampled when D was trained. Sampling is random, so differences of a few points are
noise.

In [ ]:
saved_d = project_path("eval/results/gen_D/gen_D_1.5b_summary.json")
if os.path.exists(saved_d):
    d = json.load(open(saved_d))
    cols = [("A seen", d["comparison_A"]["seen"]), ("D seen", d["samples"]["seen"]), ("E seen", reports_e["seen"]),
            ("A held", d["comparison_A"]["heldout"]), ("D held", d["samples"]["heldout"]),
            ("E held", reports_e["heldout"])]
    keys = ("four_lines", "end_word_hit", "couplets_rhyme", "aabb", "syllables", "lexical", "trait_mention", "distinct_2")
    if SMOKE_TEST:
        print("smoke test: E has only", K_PROMPTS, "prompts per set")
    print(f"{'':16s}" + "".join(f"{c:>10s}" for c, _ in cols))
    for key in keys:
        print(f"{key:16s}" + "".join(f"{r[key]:>10.1%}" for _, r in cols))
else:
    print("no saved System D results at", saved_d)

In [ ]:
summary = {"name": NAME, "model": MODEL_NAME, "stage1": STAGE1, "lr": [LR1, LR2], "max_epochs": EPOCHS,
           "patience": PATIENCE, "batch_size": BATCH_SIZE * ACCUM, "sampling": SAMPLING, "history": hist,
           "best_val_loss": {s: min(h["val_loss"] for h in v) for s, v in hist.items()}, "test_loss": test,
           "samples": reports_e}
json.dump(summary, open(os.path.join(CKPT_DIR, f"{NAME}_summary.json"), "w"), indent=2)
print("saved", os.path.join(CKPT_DIR, f"{NAME}_summary.json"),
      "- copy it and the history files to eval/results/gen_E/")